In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [ ]:
food_predictors = ['DT_FOLAC', 'DT_CALC', 'VITD_MCG', 'TOTAL_CHOLINE', 'DT_SODI', 'PRENATALYEARS','PRENATALAMOUNT','AHEI2010','AHEI_ALCDRKS','AHEI_SODIUM','AHEI_PUFAPCT','AHEI_DHAEPA','AHEI_TRFATPCT','AHEI_RMEATS','AHEI_NUTLEGS','AHEI_SUGBEVS','AHEI_WGRAINS','AHEI_FRUITS','AHEI_VEGS','DT_ALCO','DT_CAFFN','DT_FIBE','DT_SUG_T','DT_CHOL','DT_PFAT','DT_MFAT','DT_SFAT','DT_TFAT','DT_CARB','DT_KCAL', 'DT_PROT','DT_VITC', 'DT_VB12', 'DT_VITB6', 'DT_NIAC','DT_RIBO','DT_THIA', 'DT_IRON', 'DT_TOTN3']
df = pd.read_csv('../../Data/FOOD_FREQUENCY_ANALYSIS.csv', usecols=food_predictors + ['PublicID'])
# df[['PRENATALAMOUNT', 'PRENATALYEARS']] = df[['PRENATALAMOUNT', 'PRENATALYEARS']].replace({'M': np.nan, 'E': np.nan})
df[food_predictors] = df[food_predictors].apply(pd.to_numeric, errors='coerce')

In [ ]:
df_outcomes = pd.read_csv('../../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [4]:
df = pd.merge(df, df_outcomes, on='PublicID', how='inner')
df

,PublicID,DT_KCAL,DT_PROT,DT_CARB,DT_TFAT,DT_ALCO,DT_CAFFN,VITD_MCG,TOTAL_CHOLINE,DT_FOLAC,...,AHEI_SUGBEVS,AHEI_NUTLEGS,AHEI_RMEATS,AHEI_TRFATPCT,AHEI_DHAEPA,AHEI_PUFAPCT,AHEI_SODIUM,AHEI_ALCDRKS,AHEI2010,MH_outcome
0,00004O,1445.22,55.33,168.05,63.53,0.001440,1.53,2.830,270.63,96.59,...,0.000000,4.4900,6.162236,7.478604,7.360,9.020737,6.962302,2.5,52.410880,1
1,00008G,1472.37,48.61,211.97,52.04,2.330000,32.19,3.150,213.14,180.73,...,0.000000,3.3800,6.564417,7.673692,0.936,6.294495,6.954905,5.0,53.209509,0
2,00015J,2162.19,98.57,249.59,98.65,2.370000,27.46,4.840,327.76,244.97,...,5.366402,10.0000,5.310157,9.430597,5.400,10.000000,0.000000,5.0,70.084156,0
3,00016H,1014.46,32.44,101.70,50.56,12.890000,21.14,1.080,120.01,63.85,...,0.000000,0.5350,6.768916,7.119453,1.700,10.000000,9.910997,10.0,48.539366,1
4,00017F,1170.14,39.34,163.21,43.71,3.100000,24.52,4.550,164.03,193.43,...,5.133598,6.6800,10.000000,10.000000,6.400,10.000000,7.531290,5.0,76.122387,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6947,17349I,1744.90,73.89,249.98,52.68,0.002100,7.17,8.950,328.04,98.97,...,0.000000,3.1650,4.669393,8.982259,3.596,4.476044,5.179470,2.5,40.753166,1
6948,17350A,1289.42,41.81,176.94,46.82,3.970000,87.33,0.979,144.28,35.93,...,0.000000,0.4909,1.567825,7.280527,1.052,8.100696,9.633505,5.0,38.578954,1
6949,17351V,680.02,38.19,64.97,30.48,0.141000,1.79,1.570,243.00,39.22,...,9.155644,3.5790,8.588957,9.874415,3.496,6.962957,10.000000,5.0,61.258473,0
6950,17352T,1066.71,40.32,121.59,45.72,5.610000,74.00,1.380,212.77,60.02,...,0.000000,5.9600,7.600545,8.367089,9.040,8.911255,9.068236,10.0,68.488125,1


In [7]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    4165
1    2787
Name: count, dtype: int64

In [8]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    food_predictors
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.1, 'classifier__l1_ratio': 0.0}
Best Macro F1 Score: 0.5698
Model Coefficients:
num__DT_FOLAC: -0.021229304768249302
num__DT_CALC: -0.016821992943930646
num__VITD_MCG: -0.03666128030130348
num__TOTAL_CHOLINE: 0.1513310498422859
num__DT_SODI: 0.2190928249133196
num__PRENATALYEARS: -0.0774642246547086
num__PRENATALAMOUNT: -0.08262933509787265
num__AHEI2010: -0.06677235867486983
num__AHEI_ALCDRKS: -0.06990320008507118
num__AHEI_SODIUM: -0.14368208120742026
num__AHEI_PUFAPCT: 0.18394273618379664
num__AHEI_DHAEPA: 0.027266229659246837
num__AHEI_TRFATPCT: -0.045033591913782076
num__AHEI_RMEATS: 0.037630337125079884
num__AHEI_NUTLEGS: -0.0019204960012480662
num__AHEI_SUGBEVS: 0.002049070884352577
num__AHEI_WGRAINS: -0.07979987224258121
num__AHEI_FRUITS: -0.08602746072244011
num__AHEI_VEGS: -0.08538845955022906
num__DT_ALCO: 0.05675396678555939
num__DT_CAFFN: -0.044267695929071
num__DT_FIBE

In [9]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    food_predictors
)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 15, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 500}
Best Macro F1 Score: 0.5677
Feature Importances:
num__DT_FOLAC: 0.0287154551037418
num__DT_CALC: 0.025093922115407157
num__VITD_MCG: 0.024244936499376684
num__TOTAL_CHOLINE: 0.02391804456678523
num__DT_SODI: 0.023984570908200983
num__PRENATALYEARS: 0.005618387940864398
num__PRENATALAMOUNT: 0.022055901099292317
num__AHEI2010: 0.038525436825622664
num__AHEI_ALCDRKS: 0.025773204560087037
num__AHEI_SODIUM: 0.01842221913517521
num__AHEI_PUFAPCT: 0.02750544497173878
num__AHEI_DHAEPA: 0.02848779771687643
num__AHEI_TRFATPCT: 0.03182600541522815
num__AHEI_RMEATS: 0.027931866310355154
num__AHEI_NUTLEGS: 0.025909351486180045
num__AHEI_SUGBEVS: 0.017110936091941458
num__AHEI_WGRAINS: 0.029132188250180057
num__AHEI_FRUITS: 0.03162934735670252
num__AHEI_VEGS: 0.02577437731915871
num__

In [10]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    food_predictors
)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 0.9, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 4, 'classifier__n_estimators': 80, 'classifier__subsample': 0.8}
Best Macro F1 Score: 0.5790
Feature Importances:
num__DT_FOLAC: 0.02977827563881874
num__DT_CALC: 0.01854531466960907
num__VITD_MCG: 0.017601313069462776
num__TOTAL_CHOLINE: 0.022850530222058296
num__DT_SODI: 0.030141374096274376
num__PRENATALYEARS: 0.026163004338741302
num__PRENATALAMOUNT: 0.061419565230607986
num__AHEI2010: 0.054598476737737656
num__AHEI_ALCDRKS: 0.09483712911605835
num__AHEI_SODIUM: 0.030241893604397774
num__AHEI_PUFAPCT: 0.02047165296971798
num__AHEI_DHAEPA: 0.01775304414331913
num__AHEI_TRFATPCT: 0.01918533630669117
num__AHEI_RMEATS: 0.026343343779444695
num__AHEI_NUTLEGS: 0.02310536429286003
num__AHEI_SUGBEVS: 0.01859292760491371
num__AHEI_WGRAINS: 0.019424015656113625
num__AHEI_FRUITS: 0.02466844953596592
num__AHEI_V

In [11]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    food_predictors
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 0.01, 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.5695
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.5679
Precision: 0.5658
Recall: 0.5684
F1-score: 0.5626
ROC AUC: 0.5968
